# Sections 4.2–4.3 — Person Counting: Supervising Trajectories

Companion to `Chapter4_Supervising_Trajectory.ipynb`. Same three arms,
same structure, but for the three-class person-counting dataset.

Produces:

* **Table 5** — error, rescue rate, per-class recall, learned couplings
  and variances for the three supervising trajectories
* **Figure** `teacher_error_counting.pdf`
* **Figure** `teacher_confusion_counting.pdf`
* **Figure** `teacher_couplings_counting.pdf`

### What differs from the binary notebook

The state is $\{0,1,2\}$, not $\pm 1$, so the linear estimator cannot be
used — it divides by the state and is undefined at zero, which is 860 of
2163 samples. This notebook uses the **affine estimator** instead:

* joint least squares for $(c_i, b_i)$ on the regressor $[T, 1]$
* the state augmented with a constant, $A = I_2$,
  $Q = \mathrm{diag}(10, 0)$, $P_0 = \mathrm{diag}(1, 0)$
* decision by nearest admissible level rather than a sign test

The rescue rate is included here, so the counting table can carry the
same column as the binary one.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
sns.set_palette("deep")

CSV = "../data/person_counting_2163.csv"

# column names -- printed below so you can correct them on the first run
COL_EXPERT = "human_count"
COL_EVENT  = "mouse_event"
COL_MIC1   = "dBFS_mic1"
COL_MIC2   = "dBFS_mic2"

# filter configuration, Table 11
SIG_E2, C_E = 1.0, 1.0
Q_VAL, P0_VAL = 10.0, 1.0
EPS = 1e-6
LEVELS = np.array([0, 1, 2])

# Offline affine references, Table 3 (counting rows)
C_REF     = {1: 7.007,  2: 7.137}
SIGMA_REF = {1: 34.673, 2: 12.410}


## 1. Load and build ground truth

Ground truth follows Eq. (3.10), not the binary parity rule. Each button
toggles its own class: a press of code $\ell$ sets the state to $\ell$
unless the state is already $\ell$, in which case it returns to $0$.
Ticks carrying no event hold the previous state.

Both assertions must pass. If either fires, stop — every number below
depends on them.


In [2]:
df = pd.read_csv(CSV)
print("columns found:", list(df.columns))

df = df[[COL_EXPERT, COL_EVENT, COL_MIC1, COL_MIC2]].dropna().reset_index(drop=True)

# Event-state reconstruction, Section 3.4.2: each button toggles its own class
ev = df[COL_EVENT].astype(int).values
X = np.zeros(len(ev), dtype=int)
prev = 0
for k, e in enumerate(ev):
    if e in (1, 2):
        prev = 0 if prev == e else e
    X[k] = prev

N = len(X)
y_e = np.clip(df[COL_EXPERT].astype(float).values, 0, 2)   # Expert sensor output, Section 3.4.1
y1  = df[COL_MIC1].astype(float).values                    # raw dBFS, NOT centred
y2  = df[COL_MIC2].astype(float).values

counts = tuple(int((X == c).sum()) for c in LEVELS)
expert_alone = np.clip(np.rint(y_e), 0, 2).astype(int)

print(f"N = {N}                       (Table 1 expects 2163)")
print(f"class counts = {counts}   (Table 1 expects (860, 559, 744))")
print(f"expert alone error = {100*(expert_alone != X).mean():.2f}%   "
      f"(Table 1 expects 44.24%)")

assert N == 2163, "row count does not match Table 1 -- stop here"
assert counts == (860, 559, 744), "class counts do not match -- stop here"


columns found: ['elapsed_seconds', 'RMS_mic1', 'dBFS_mic1', 'RMS_mic2', 'dBFS_mic2', 'human_count', 'mouse_event']
N = 2163                       (Table 1 expects 2163)
class counts = (860, 559, 744)   (Table 1 expects (860, 559, 744))
expert alone error = 44.24%   (Table 1 expects 44.24%)


## 2. The affine estimator and the filter

`affine_fit` implements Eq. (3.15). The regressor matrix is rank-deficient
until the supervising trajectory has visited two distinct levels; until
then the previous estimates are held, as Section 3.8 specifies.

`run` is the counting analogue of the binary notebook's `run`. The state
is augmented with a constant so the offset can be carried through the
Kalman recursion, and the constant component is reset to unity after each
update to prevent drift.


In [3]:
def affine_fit(y_hist, T_hist, prev_c, prev_b):
    """Eq. (3.15): joint LS for (c, b) on the regressor [T, 1].
    Holds the previous estimates while T has visited fewer than two levels."""
    if len(np.unique(T_hist)) < 2:
        return prev_c, prev_b
    Phi = np.column_stack([T_hist, np.ones_like(T_hist)])
    sol = np.linalg.pinv(Phi.T @ Phi) @ (Phi.T @ y_hist)
    return float(sol[0]), float(sol[1])


def run(teacher_mode, learn=True):
    """Expert + two inexpert sensors, augmented state (Eq. 3.17)."""
    A = np.eye(2)
    Q = np.diag([Q_VAL, 0.0])
    Xt = np.array([0.0, 1.0])              # [state; constant]
    P  = np.diag([P0_VAL, 0.0])

    c1h = c2h = 1.0
    b1h = b2h = 0.0
    s1h = s2h = 1.0

    dec, exp_dec = np.zeros(N, dtype=int), np.zeros(N, dtype=int)
    hist = {k: np.zeros(N) for k in ("c1", "c2", "b1", "b2", "s1", "s2")}

    Xe, Pe = 0.0, P0_VAL                   # expert-only filter, scalar

    for k in range(N):
        # ---- expert-only filter (never sees the microphones) ----
        Pe_m = Pe + Q_VAL
        Ke = Pe_m * C_E / (C_E * Pe_m * C_E + SIG_E2)
        Xe = Xe + Ke * (y_e[k] - C_E * Xe)
        Pe = (1 - Ke * C_E) * Pe_m
        exp_dec[k] = int(np.clip(round(Xe), 0, 2))

        # ---- fused filter ----
        C = np.array([[C_E, 0.0],
                      [c1h, b1h],
                      [c2h, b2h]])
        R = np.diag([SIG_E2, max(s1h, EPS), max(s2h, EPS)])

        Xm = A @ Xt
        Pm = A @ P @ A.T + Q
        resid = np.array([y_e[k], y1[k], y2[k]]) - C @ Xm
        K  = Pm @ C.T @ np.linalg.inv(C @ Pm @ C.T + R)
        Xt = Xm + K @ resid
        P  = (np.eye(2) - K @ C) @ Pm
        Xt[1] = 1.0                        # reset the constant component

        dec[k] = int(LEVELS[np.argmin(np.abs(LEVELS - Xt[0]))])

        # ---- parameter update ----
        if learn:
            if   teacher_mode == "gt":     T = X[:k+1].astype(float)
            elif teacher_mode == "expert": T = exp_dec[:k+1].astype(float)
            elif teacher_mode == "fused":  T = dec[:k+1].astype(float)
            else: raise ValueError(teacher_mode)

            c1h, b1h = affine_fit(y1[:k+1], T, c1h, b1h)
            c2h, b2h = affine_fit(y2[:k+1], T, c2h, b2h)
            s1h = float(np.var(y1[:k+1] - b1h - c1h * T))
            s2h = float(np.var(y2[:k+1] - b2h - c2h * T))

        for key, val in zip(("c1", "c2", "b1", "b2", "s1", "s2"),
                            (c1h, c2h, b1h, b2h, s1h, s2h)):
            hist[key][k] = val

    return dec, hist


def recall(pred):
    return [100 * ((pred == c) & (X == c)).sum() / max((X == c).sum(), 1)
            for c in LEVELS]


def confusion(pred):
    M = np.zeros((3, 3))
    for i in LEVELS:
        for j in LEVELS:
            M[i, j] = ((X == i) & (pred == j)).sum()
    return M / np.maximum(M.sum(axis=1, keepdims=True), 1)


## 3. Manuscript Table 5

The rescue rate is computed over the samples on which the expert sensor
is wrong — the only regime in which the inexpert measurements can change
the outcome. It does **not** by itself separate the microphones'
contribution from the filter's temporal smoothing; the frozen-parameter
control is what does that.


In [4]:
ARMS  = {"gt": "Annotated", "expert": "Expert-only", "fused": "Fused"}
STYLE = {"gt": "seagreen",  "expert": "firebrick",   "fused": "steelblue"}

wrong = (expert_alone != X)
n_wrong = int(wrong.sum())

res = {}
print("=" * 88)
print(f"{'Trajectory':<13}{'Error':>8}{'Rescue':>9}{'Recall 0/1/2':>24}"
      f"{'c1':>9}{'c2':>9}{'s1':>9}{'s2':>9}")
print("-" * 88)
print(f"{'Expert alone':<13}{100*(expert_alone != X).mean():>7.2f}%{'---':>9}"
      f"{'/'.join(f'{r:.1f}%' for r in recall(expert_alone)):>24}"
      f"{'---':>9}{'---':>9}{'---':>9}{'---':>9}")

for mode, label in ARMS.items():
    dec, hist = run(mode)
    res[mode] = dict(dec=dec, hist=hist,
                     err=100 * (dec != X).mean(),
                     rescue=100 * ((dec == X) & wrong).sum() / n_wrong,
                     cum=100 * np.cumsum(dec != X) / (np.arange(N) + 1),
                     cm=confusion(dec))
    print(f"{label:<13}{res[mode]['err']:>7.2f}%{res[mode]['rescue']:>8.2f}%"
          f"{'/'.join(f'{r:.1f}%' for r in recall(dec)):>24}"
          f"{hist['c1'][-1]:>9.3f}{hist['c2'][-1]:>9.3f}"
          f"{hist['s1'][-1]:>9.3f}{hist['s2'][-1]:>9.3f}")

print("-" * 88)
print(f"references:  c1* = {C_REF[1]:.3f}  c2* = {C_REF[2]:.3f}"
      f"   s1* = {SIGMA_REF[1]:.3f}  s2* = {SIGMA_REF[2]:.3f}")
print(f"rescue rate computed over {n_wrong} camera-wrong samples "
      f"({100*n_wrong/N:.2f}% of {N})")
print("\ncheck: errors should read 12.34% / 38.56% / 49.70%")
print("=" * 88)


Trajectory      Error   Rescue            Recall 0/1/2       c1       c2       s1       s2
----------------------------------------------------------------------------------------
Expert alone   44.24%      ---      100.0%/31.1%/23.1%      ---      ---      ---      ---


Annotated      12.34%   76.38%       98.8%/86.8%/75.4%    7.007    7.137   34.673   12.410


Expert-only    38.56%   15.15%       99.7%/46.2%/28.8%    4.013    3.607   64.882   45.140


Fused          49.70%   18.08%        99.1%/41.7%/0.4%   15.579    6.634   14.498   39.801
----------------------------------------------------------------------------------------
references:  c1* = 7.007  c2* = 7.137   s1* = 34.673  s2* = 12.410
rescue rate computed over 957 camera-wrong samples (44.24% of 2163)

check: errors should read 12.34% / 38.56% / 49.70%


## 4. Figures


In [5]:
# ---- cumulative error ----
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(100 * np.cumsum(expert_alone != X) / (np.arange(N) + 1),
        color="black", ls="--", lw=1.5,
        label=f"Expert sensor alone ({100*(expert_alone != X).mean():.2f}%)")
for mode, label in ARMS.items():
    ax.plot(res[mode]["cum"], color=STYLE[mode], lw=1.8,
            label=f"{label} ({res[mode]['err']:.2f}%)")
ax.set_xlabel("Time step $k$")
ax.set_ylabel("Cumulative classification error [%]")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("teacher_error_counting.pdf")
plt.show()

# ---- confusion matrices ----
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, (mode, label) in zip(axes, ARMS.items()):
    M = res[mode]["cm"]
    ax.grid(False)                      # <-- add this
    ax.imshow(M, cmap="Blues", vmin=0, vmax=1)
    for i in LEVELS:
        for j in LEVELS:
            ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center",
                    fontsize=9, color="white" if M[i, j] > 0.5 else "black")
    ax.set_xticks(LEVELS); ax.set_yticks(LEVELS)
    ax.set_xlabel("Predicted")
    ax.set_title(f"{label} ({res[mode]['err']:.2f}%)")
axes[0].set_ylabel("True")
fig.tight_layout()
fig.savefig("teacher_confusion_counting.pdf")
plt.show()

# ---- learned couplings ----
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for idx, (ax, key) in enumerate(zip(axes, ("c1", "c2")), start=1):
    ax.axhline(C_REF[idx], color="black", ls="--", lw=1.4,
               label=rf"Offline reference $c_{idx}^*$ = {C_REF[idx]:.3f}")
    for mode, label in ARMS.items():
        ax.plot(res[mode]["hist"][key], color=STYLE[mode], lw=1.8,
                label=f"{label} ({res[mode]['hist'][key][-1]:.3f})")
    ax.set_ylabel(rf"$\hat{{c}}_{idx}(k)$")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8, ncol=2)
axes[1].set_xlabel("Time step $k$")
fig.tight_layout()
fig.savefig("teacher_couplings_counting.pdf")
plt.show()

print("saved: teacher_error_counting.pdf, teacher_confusion_counting.pdf,")
print("       teacher_couplings_counting.pdf")


/tmp/ipykernel_902/4093446745.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_902/4093446745.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: teacher_error_counting.pdf, teacher_confusion_counting.pdf,
       teacher_couplings_counting.pdf


/tmp/ipykernel_902/4093446745.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Convergence check

The mechanism section claims the fused arm *converges* to a wrong
calibration rather than diverging. This is what supports that claim: if
the change over the final quarter is small relative to the distance from
the reference, the estimate has settled.


In [6]:
print("fused arm, final quarter of the recording:")
for key in ("c1", "c2"):
    h = res["fused"]["hist"][key]
    q = h[int(0.75 * N):]
    idx = 1 if key == "c1" else 2
    print(f"  {key}: final {h[-1]:8.3f}   reference {C_REF[idx]:6.3f}   "
          f"change over last quarter {h[-1] - h[int(0.75*N)]:+7.3f}   "
          f"range [{q.min():.3f}, {q.max():.3f}]")


fused arm, final quarter of the recording:
  c1: final   15.579   reference  7.007   change over last quarter  -0.226   range [15.552, 15.870]
  c2: final    6.634   reference  7.137   change over last quarter  -0.441   range [6.601, 7.368]


## 6. Bucket means (optional)

Supports the selection argument in the mechanism section. If the
empirical mean of each predicted bucket differs from what the confusion
matrix alone predicts — higher for $T = 1$, lower for $T = 0$ — then the
fused trajectory is selecting its own labels, and that is what inflates
the coupling beyond label compression alone.


In [7]:
mu = {0: -40.33, 1: -31.56, 2: -26.39}     # Table 2, microphone 1
dec, cm = res["fused"]["dec"], res["fused"]["cm"]

print(f"{'bucket':>8}{'empirical':>12}{'no selection':>14}{'difference':>12}")
for l in LEVELS:
    if (dec == l).sum() == 0:
        continue
    w = np.array([cm[t, l] * (X == t).sum() for t in LEVELS])
    pred = sum(w[t] * mu[t] for t in LEVELS) / w.sum()
    emp = y1[dec == l].mean()
    print(f"{f'T={l}':>8}{emp:>12.2f}{pred:>14.2f}{emp - pred:>+12.2f}")


  bucket   empirical  no selection  difference
     T=0      -38.85        -36.13       -2.72
     T=1      -23.22        -28.10       +4.88
     T=2      -12.75        -26.39      +13.64
